# Stage 3: Model Evaluation

DVC stage `evaluate`. Scores the trained model on the held-out test set and writes:

- `metrics/metrics.json` — tracked with `dvc metrics show` / `dvc metrics diff`
- `plots/roc_curve.csv` — tracked with `dvc plots show`
- `plots/confusion_matrix.png` and `plots/feature_importance.png`

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

## Load parameters, model and test data

In [ ]:
with open('params.yaml') as f:
    params = yaml.safe_load(f)

target = params['base']['target']
threshold = params['evaluate']['threshold']

bundle = joblib.load('models/model.pkl')
model, features = bundle['model'], bundle['features']

test_df = pd.read_csv(Path(params['data']['processed_dir']) / 'test.csv')
X_test = test_df[features]
y_test = test_df[target]
print(f'X_test: {X_test.shape}, threshold: {threshold}')

## Predict and compute metrics

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= threshold).astype(int)

metrics = {
    'accuracy': round(float(accuracy_score(y_test, pred)), 4),
    'precision': round(float(precision_score(y_test, pred)), 4),
    'recall': round(float(recall_score(y_test, pred)), 4),
    'f1': round(float(f1_score(y_test, pred)), 4),
    'roc_auc': round(float(roc_auc_score(y_test, proba)), 4),
    'test_rows': int(len(y_test)),
}

Path('metrics').mkdir(exist_ok=True)
with open('metrics/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

## ROC curve data

In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)
Path('plots').mkdir(exist_ok=True)
roc = pd.DataFrame({'fpr': fpr, 'tpr': tpr})
step = max(1, len(roc) // 500)
roc.iloc[::step].round(5).to_csv('plots/roc_curve.csv', index=False)
print(f'Saved {len(roc.iloc[::step])} ROC points')

## Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
            xticklabels=['Not cancelled', 'Cancelled'], yticklabels=['Not cancelled', 'Cancelled'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (test set)')
fig.tight_layout()
fig.savefig('plots/confusion_matrix.png', dpi=120)
plt.close(fig)
cm

## Feature importance

In [ ]:
importance = (pd.DataFrame({'feature': features, 'importance': model.feature_importances_})
              .sort_values('importance', ascending=False)
              .reset_index(drop=True))
top = importance.head(15)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top['feature'][::-1], top['importance'][::-1], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Top 15 Feature Importances')
fig.tight_layout()
fig.savefig('plots/feature_importance.png', dpi=120)
plt.close(fig)
importance.head(10)